In [1]:
## Load prerequisites libraries
import numpy as np
import pandas as pd
import warnings #Possibly not required
import json
import arrow #Possibly not required
import pyarrow as pa
import pyarrow.parquet as pq


In [2]:
# Check lims dsn connection to database before proceeding
def get_lims_connection(db_config):
    global odbcLimsConnection
    if db_config['dbType'] == "oracle":
        import oracledb
        oracledb.init_oracle_client(config.lib_dir)
        odbcLimsConnection = oracledb.connect(user=db_config['limsdb_username'], 
                                            password=db_config['limsdb_password'],
                                            dsn=db_config['limsdb_dsn'])
        odbcLimsConnection.arraysize = 5000
        
    elif db_config['dbType']== "postgres":
        import psycopg2
        odbcLimsConnection = psycopg2.connect('dbname=' + db_config['limsdb_database'] + ' user=' + db_config['limsdb_username'] + 
                                            ' password=' + db_config['limsdb_password'] + ' host=' + db_config['limsdb_hostname'])

    else:
        import pyodbc
        odbcLimsConnection = pyodbc.connect('DRIVER=' + db_config['limsdb_driver'] + ';DATABASE=' + db_config['limsdb_database'] +
                                        ';UID=' + db_config['limsdb_username'] + ';PWD=' + db_config['limsdb_password']+
                                        ';SERVER=' + db_config['limsdb_hostname'] + ';PORT=' + db_config['limsdb_port'] +
                                        ';Trusted_Connection=yes')
    return odbcLimsConnection



In [3]:
import json

def load_config(config_path):
    with open(config_path, 'r') as file:
        config = json.load(file)
    return config

In [4]:
def run_pipeline_for_database(db_config):
    print(f"Running pipeline for {db_config['customerName']} at {db_config['limsdb_hostname']}")
    # Here you would add your data engineering pipeline logic
    # This could involve connecting to the database, processing data, etc.
    get_lims_connection(db_config)
    rawDataFrame = pd.read_sql('''
                               
                  select counter, mlar_type, audit_table_name, audit_table_key, audit_timestamp , audit_action , audit_counter , audit_type , mean_length, severity  from t0083_ar ta 
                               
                 
                 ''',con=odbcLimsConnection)
        # Tag with source DB
    rawDataFrame['SOURCE_DB'] = db_config['customerName']

    # Convert to table for parquet conversion
    table = pa.Table.from_pandas(rawDataFrame)

    # Convert to parquet
    fileName = 'C:/MLAR_DATA/T0001/labelled_anomalies_'+ db_config["customerName"] +'.parquet'
    pq.write_table(table,fileName)

    

In [5]:

def main():
    config = load_config("config_db.json")
    for db in config['databases']:
        run_pipeline_for_database(db)

if __name__ == "__main__":
    main()

Running pipeline for X5 at lwanalytics-dev.cwrjruraj02j.us-east-1.rds.amazonaws.com
Running pipeline for X8 at lwanalytics-dev.cwrjruraj02j.us-east-1.rds.amazonaws.com


C:\Users\dan.larson\AppData\Local\Temp\4\ipykernel_10972\3830542404.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rawDataFrame = pd.read_sql('''
C:\Users\dan.larson\AppData\Local\Temp\4\ipykernel_10972\3830542404.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rawDataFrame = pd.read_sql('''


Running pipeline for X7 at lwanalytics-dev.cwrjruraj02j.us-east-1.rds.amazonaws.com


C:\Users\dan.larson\AppData\Local\Temp\4\ipykernel_10972\3830542404.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rawDataFrame = pd.read_sql('''


In [6]:
import duckdb as db

db.sql("select SOURCE_DB, count(*) from read_parquet('C:/MLAR_DATA/T0001/labelled_*.parquet') group by SOURCE_DB")


┌───────────┬──────────────┐
│ SOURCE_DB │ count_star() │
│  varchar  │    int64     │
├───────────┼──────────────┤
│ X5        │         6303 │
│ X8        │         5313 │
│ X7        │         9553 │
└───────────┴──────────────┘

In [7]:
import duckdb as db

db.sql("select SOURCE_DB, case when severity in ('2','3','4','5') then 1 else 0 end as anomaly, count(*) from read_parquet('C:/MLAR_DATA/T0001/labelled_*.parquet') group by 1,2 order by 1, 2")

┌───────────┬─────────┬──────────────┐
│ SOURCE_DB │ anomaly │ count_star() │
│  varchar  │  int32  │    int64     │
├───────────┼─────────┼──────────────┤
│ X5        │       0 │         1827 │
│ X5        │       1 │         4476 │
│ X7        │       0 │         4268 │
│ X7        │       1 │         5285 │
│ X8        │       0 │         5313 │
└───────────┴─────────┴──────────────┘